# Agent Memory for LLM Agents

This notebook extends the study course with memory systems. The goal is to make memory concrete: what gets stored, how it is retrieved, when it should be updated, and why careless memory design can hurt an otherwise good agent.

## Learning goals

- Understand why agent systems need memory beyond a single context window.
- Distinguish short-term, long-term, episodic, semantic, and vector memory patterns.
- Experiment with memory retrieval and memory update strategies.
- See how memory can connect to the existing workflow without hiding logic inside the notebook.


## Concept explanation

As in the other notebooks, we begin by confirming the active Python executable. This makes it easy to verify that Jupyter is using the uv-managed kernel instead of a random system interpreter.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(sys.executable)

This setup cell normalizes the working directory and imports the reusable memory classes and helpers from `src/`. The notebook remains educational, but the real logic still lives in the package.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.config import get_paths
from src.ingestion import build_demo_index
from src.memory import (
    LongTermMemory,
    ShortTermMemory,
    VectorMemory,
    build_memory_augmented_context,
    display_memory,
    score_memory_importance,
    selective_memory_update,
    summarize_memory_events,
)
from src.utils import display_trace
from src.workflow import run_workflow_with_memory

pd.set_option('display.max_colwidth', 140)
paths = get_paths()
memory_store_path = paths.logs_dir / 'agent_memory_notebook_store.json'
if memory_store_path.exists():
    memory_store_path.unlink()


## Introduction

Agents need memory because a single context window is not enough for every interaction. Stateless pipelines answer only from the current prompt and retrieved context. Memory-enabled agents can remember what happened earlier, reuse learned facts, and personalize future responses.

Context-window limits create three practical problems:

- old but important information falls out of scope
- repeated user preferences are forgotten
- long-running tasks lose continuity across turns

It helps to separate memory types:

- short-term memory: the active working buffer for recent turns and reasoning state
- long-term memory: persisted facts, preferences, and durable knowledge
- episodic memory: records of specific past interactions or events
- semantic memory: generalized facts or concepts extracted from many interactions

Vector memory is a retrieval mechanism often used for episodic or semantic memory. Instead of exact keys, it retrieves memories by similarity.


## Implementation

### Short-Term Memory

Short-term memory is the agent's working scratchpad. Common examples are a conversation buffer, temporary reasoning state, or the last few tool outputs. The class below keeps a rolling list of events, lets us inspect the last `N`, and supports clearing the buffer.


In [ ]:
short_term = ShortTermMemory(max_items=6)
short_term.append_event('user', 'Can you explain the rollout timeline?')
short_term.append_event('assistant', 'The pilot runs from March 10, 2025 to April 4, 2025.')
short_term.append_event('user', 'Please keep future answers concise.')
short_term.append_event('assistant', 'Noted. I will keep answers concise when possible.')
short_term.to_frame()


This small experiment simulates a few conversation turns and then asks for only the most recent memory items. That mirrors how an agent might keep a working context without carrying the whole conversation forever.


In [ ]:
recent_turns = short_term.last_n(3)
pd.DataFrame(recent_turns)


### Long-Term Memory

Long-term memory stores durable information such as user preferences, stable facts, or persistent project knowledge. In this project we use simple JSON persistence so the storage format stays transparent and easy to inspect.


In [ ]:
long_term = LongTermMemory(memory_store_path)
long_term.store_memory('mina_answer_style', 'Mina prefers concise answers that lead with exact dates.', category='preference')
long_term.store_memory('apollo_pilot_city', 'Team Apollo is piloting the scheduling template in Seoul.', category='fact')
reloaded_long_term = LongTermMemory(memory_store_path)
reloaded_long_term.to_frame()


Reloading the store from disk is important because durable memory should survive beyond one notebook cell or one process. This cell checks that the JSON-backed store can round-trip cleanly.


In [ ]:
reloaded_long_term.retrieve_memory('mina_answer_style')


### Vector Memory

Vector memory supports semantic retrieval. Instead of asking for an exact key, we encode memory text into vectors and search by similarity. This is useful when the user asks for something related to a previous preference or event, but does not use the same words.


In [ ]:
vector_memory = VectorMemory()
vector_memory.add_memory('Mina prefers concise rollout answers that begin with the exact launch date.', metadata={'kind': 'preference'}, importance=0.9)
vector_memory.add_memory('The rollout FAQ should mention the May 5, 2025 launch date.', metadata={'kind': 'fact'}, importance=0.8)
vector_memory.add_memory('The pilot retrospective is scheduled for June.', metadata={'kind': 'event'}, importance=0.5)
vector_search_results = vector_memory.search('How should I answer rollout timing for Mina?', top_k=3)
pd.DataFrame(vector_search_results)


## Memory Retrieval in Agents

When an agent reasons, it often needs two retrieval passes:

1. retrieve external evidence such as documents
2. retrieve internal evidence such as relevant memories

A simple text architecture is: `query -> retrieve docs -> retrieve memories -> combine context -> answer`. The next cell uses the existing document retriever and adds memory retrieval on top of it.


In [ ]:
retriever = build_demo_index(persist=False)
retrieved_docs = retriever.search('When does the organization-wide rollout begin?', top_k=3)
context_bundle = build_memory_augmented_context(
    query='How should I answer rollout timing for Mina?',
    retrieved_docs=retrieved_docs,
    short_term_memory=short_term,
    long_term_memory=reloaded_long_term,
    vector_memory=vector_memory,
    top_k=3,
)
{
    'doc_sources': [doc['source'] for doc in context_bundle['retrieved_docs']],
    'memory_count': len(context_bundle['retrieved_memories']),
    'memory_texts': [memory['text'] for memory in context_bundle['retrieved_memories']],
}


### Memory Update Strategy

Storing everything is a bad memory policy. It increases noise, slows retrieval, and makes it harder to find the truly useful facts later. Common strategies include append-only logging, summarization of recent history, and selective storage based on importance.

This repository uses a simple rule-based importance score so you can see the update decision instead of hiding it inside a model.


In [ ]:
candidates = [
    'Remember: Mina prefers concise rollout summaries with dates first.',
    'The user said thanks.',
    'Important: Team Apollo is piloting the scheduling template in Seoul.',
]
importance_frame = pd.DataFrame(
    {
        'candidate': candidates,
        'importance_score': [score_memory_importance(text, {'category': 'fact', 'source': 'user'}) for text in candidates],
    }
)
importance_frame


This update experiment stores only memories that cross the threshold. The goal is to make the storage policy inspectable, so you can see why some items are kept and others are dropped.


In [ ]:
update_decisions = [
    selective_memory_update(
        text='Remember: Mina prefers concise rollout summaries with dates first.',
        key='mina_style_rule',
        long_term_memory=reloaded_long_term,
        vector_memory=vector_memory,
        threshold=0.55,
        category='preference',
        metadata={'source': 'user'},
    ),
    selective_memory_update(
        text='The user said thanks.',
        key='low_signal_event',
        long_term_memory=reloaded_long_term,
        vector_memory=vector_memory,
        threshold=0.55,
        category='event',
        metadata={'source': 'user'},
    ),
]
pd.DataFrame(update_decisions)


## Memory in the Agent Workflow

Memory can be integrated into the workflow without redesigning the whole project. In this repository the optional memory-aware runner retrieves memories, records them in the trace, and can update short-term or persistent stores after the answer is produced.

The example flow is: `query -> retrieve docs -> retrieve memory -> generate answer -> update memory`. The notebook keeps this conservative: memories are observable and traceable, rather than silently changing every answer.


In [ ]:
memory_state = run_workflow_with_memory(
    'When does the organization-wide rollout begin?',
    retriever=retriever,
    short_term_memory=short_term,
    long_term_memory=reloaded_long_term,
    vector_memory=vector_memory,
    include_memories_in_context=False,
    update_memory=True,
)
{
    'final_status': memory_state['final_status'],
    'final_answer': memory_state['final_answer'],
    'retrieved_memories': len(memory_state['retrieved_memories']),
    'memory_updates': len(memory_state['memory_updates']),
}


## Visualization

Memory debugging is important because it is easy to accumulate hidden state accidentally. A good visualization shows what is in short-term memory, what was persisted to long-term memory, and what is available for vector search.


In [ ]:
display_memory(
    short_term_memory=short_term,
    long_term_memory=reloaded_long_term,
    vector_memory=vector_memory,
)


This trace view shows where memory retrieval and memory updates appear in the workflow. The extra nodes make memory behavior inspectable instead of magical.


In [ ]:
display_trace(memory_state['trace'])


## Experiment

### Experiment 1: memory improves answer relevance

Without memory, the agent only sees documents. With memory, it can also recover the user's preferred answer style or a prior fact that matters to the current task. This cell compares the two context bundles side by side.


In [ ]:
query = 'How should I answer rollout timing for Mina?'
no_memory_bundle = build_memory_augmented_context(query=query, retrieved_docs=retrieved_docs, top_k=3)
with_memory_bundle = build_memory_augmented_context(
    query=query,
    retrieved_docs=retrieved_docs,
    short_term_memory=short_term,
    long_term_memory=reloaded_long_term,
    vector_memory=vector_memory,
    top_k=3,
)
pd.DataFrame(
    [
        {
            'mode': 'docs_only',
            'memory_count': len(no_memory_bundle['retrieved_memories']),
            'combined_context': str(no_memory_bundle['combined_context']),
        },
        {
            'mode': 'docs_plus_memory',
            'memory_count': len(with_memory_bundle['retrieved_memories']),
            'combined_context': str(with_memory_bundle['combined_context']),
        },
    ]
)


### Experiment 2: memory pollution

Memory can become noisy if we keep everything. To illustrate that risk, this cell adds several distracting memories and then reruns similarity search. Notice how irrelevant but overlapping items can crowd the ranking.


In [ ]:
polluted_memory = VectorMemory()
polluted_memory.add_memory('Mina prefers concise rollout answers that begin with the exact launch date.', metadata={'kind': 'preference'}, importance=0.9)
polluted_memory.add_memory('Mina is ordering snacks for the rollout celebration.', metadata={'kind': 'noise'}, importance=0.4)
polluted_memory.add_memory('Rollout posters should use the coral brand palette.', metadata={'kind': 'noise'}, importance=0.4)
polluted_memory.add_memory('The rollout FAQ should mention the May 5, 2025 launch date.', metadata={'kind': 'fact'}, importance=0.8)
pd.DataFrame(polluted_memory.search('How should I answer rollout timing for Mina?', top_k=4))


### Experiment 3: memory summarization

Summarization is one way to compress recent history instead of storing every turn forever. Here we collapse the recent short-term buffer into a single summary string that could be promoted into longer-term storage.


In [ ]:
summary_text = summarize_memory_events(short_term.events, limit=5)
reloaded_long_term.store_memory('recent_memory_summary', summary_text, category='summary')
pd.DataFrame(
    {
        'summary_text': [summary_text],
        'stored_summary': [reloaded_long_term.retrieve_memory('recent_memory_summary')['value']],
    }
)


## Result analysis

The notebook shows that memory is helpful when it is selective, retrievable, and inspectable. It also shows the main risks: noisy memories can pollute retrieval, and hidden updates can make debugging hard.

The table below summarizes the most important observations from the experiments.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {
            'theme': 'short_term_memory',
            'observation': f"buffer size after workflow run: {len(short_term.events)} events",
        },
        {
            'theme': 'long_term_memory',
            'observation': f"persisted items: {len(reloaded_long_term.list_items())}",
        },
        {
            'theme': 'vector_memory',
            'observation': f"top memory hit: {vector_search_results[0]['text'] if vector_search_results else 'none'}",
        },
        {
            'theme': 'workflow_integration',
            'observation': f"trace contains memory nodes: {any(entry['node'] == 'retrieve_memories' for entry in memory_state['trace'])}",
        },
    ]
)
analysis_frame


## Takeaways

- Memory helps agents preserve continuity, personalization, and durable facts.
- Short-term memory is useful for active context, while long-term and vector memory support persistence and retrieval.
- Selective storage matters because noisy memory can damage relevance.
- Memory should be visible in traces and tables so you can debug it.
- Future improvements could include better summarization, recency-aware ranking, and separate stores for preferences versus factual memory.
